<a href="https://colab.research.google.com/github/gregblast-bot/AutoMagic_Detection/blob/main/MagicCardDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MAGIC CARD ORGANIZER TRAINING**
We are going to try to train the data on google collab because why not learn a new skill today.

Below is the definition of functions

# Functions

In [ ]:
# ------------------------------------------------
# --- IMPORTS
# ------------------------------------------------
import os
import sys
import requests
import time
import math
import random
from pathlib import Path
from io import BytesIO
from PIL import Image
import numpy as np
import cv2
import shutil
from IPython.display import display, Javascript
import matplotlib.pyplot as plt
from google.colab.output import eval_js

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")

# ------------------------------------------------
# --- CONFIG
# ------------------------------------------------
NUM_CARD_IMAGES = 10    # Scryfall Samples
NUM_OTHER_IMAGES = 10   # Images of non-magic the gathering cards
IMAGE_SIZE = 96         # All images will be resized to this size
BATCH_SIZE = 32         # Number of images to train on at once
NUM_CHANNELS = 1

# ------------------------------------------------
# --- PATHS
# ------------------------------------------------
DATA_DIR = Path("/content/mtg_dataset")
CARD_DIR = DATA_DIR / "card"
NOT_CARD_DIR = DATA_DIR / "not_card"
MODEL_DIR = Path("/content/model_output")
AUGMENTED_CARD_DIR = DATA_DIR / "augmented"


# ------------------------------------------------
# --- FUNCTIONS
# ------------------------------------------------
def fetch_scryfall_cards(num_cards=NUM_CARD_IMAGES):
  """
  Download random MTG card images from Scryfall bulk data
  """

  print(f"Fetching {num_cards} card images from Scryfall...")
  downloaded = 0
  errors = 0
  max_errors = 50  # Give up after this many consecutive failures

  while downloaded < num_cards and errors < max_errors:
        try:
            resp = requests.get(
                "https://api.scryfall.com/cards/random",
                headers={"User-Agent": "MTGCardDetector/1.0"},
                timeout=10,
            )
            resp.raise_for_status()
            card = resp.json()

            # Get the small image
            img_url = card.get("image_uris", {}).get("small")
            if not img_url:
                # Some cards that are double-faced have images nested differently
                faces = card.get("card_faces", [])
                if faces:
                    img_url = faces[0].get("image_uris", {}).get("small")

            if not img_url:
                errors += 1
                continue

            # Download the image
            img_resp = requests.get(img_url, timeout=10)
            img_resp.raise_for_status()

            img = Image.open(BytesIO(img_resp.content))
            img.save(CARD_DIR / f"card_{downloaded:04d}.jpg")
            downloaded += 1
            errors = 0

            if downloaded % 100 == 0:
                print(f"  Downloaded {downloaded}/{num_cards} cards")

            # Wait for scryfall
            time.sleep(0.5)

        except Exception as e:
            errors += 1
            if errors % 10 == 0:
                print(f"  Warning: {errors} errors so far. Last: {e}")
            time.sleep(0.5)

  print(f"Downloaded {downloaded} card images ({errors} final error count)")
  return downloaded

def fetch_non_card_images(num_images=NUM_OTHER_IMAGES):
    """
    Download random non-card images for negative samples.
    """

    print(f"Fetching {num_images} non-card images...")
    downloaded = 0
    errors = 0
    max_errors = 50

    while downloaded < num_images and errors < max_errors:
        try:
            # Picsum returns a random image at the requested size
            resp = requests.get(
                f"https://picsum.photos/146/204",  # Similar aspect ratio to MTG cards
                timeout=10,
            )
            resp.raise_for_status()

            img = Image.open(BytesIO(resp.content)).convert("RGB")
            img.save(NOT_CARD_DIR / f"other_{downloaded:04d}.jpg")
            downloaded += 1
            errors = 0

            if downloaded % 50 == 0:
                print(f"  Downloaded {downloaded}/{num_images} non-card images")

            time.sleep(0.3)

        except Exception as e:
            errors += 1
            if errors % 10 == 0:
                print(f"  Warning: {errors} errors so far. Last: {e}")
            time.sleep(0.5)

    print(f"Downloaded {downloaded} non-card images ({errors} final error count)")
    return downloaded

In [ ]:

# ------------------------------------------------
# --- AUGMENTATION PARAMETERS
# ------------------------------------------------
PERSPECTIVE_INTENSITY = 0.3
BRIGHTNESS_MAX_DELTA = 0.3
BLUR_MAX_KERNEL = 7
NOISE_FACTOR = 0.05


# ------------------------------------------------
# --- AUGMENTATION FUNCTIONS
# ------------------------------------------------
def perspective_warp_cv(image_np):
    h, w = image_np.shape[:2]
    max_shift = int(min(h, w) * PERSPECTIVE_INTENSITY)
    if max_shift < 2:
        return image_np

    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])

    def _shift():
        return random.randint(max_shift // 2, max_shift)

    style = random.choice(["top_narrow", "bottom_narrow", "left_narrow", "right_narrow"])

    if style == "top_narrow":
        dst = np.float32([[_shift(), _shift()], [w - _shift(), _shift()], [w, h], [0, h]])
    elif style == "bottom_narrow":
        dst = np.float32([[0, 0], [w, 0], [w - _shift(), h - _shift()], [_shift(), h - _shift()]])
    elif style == "left_narrow":
        dst = np.float32([[_shift(), _shift()], [w, 0], [w, h], [_shift(), h - _shift()]])
    else:
        dst = np.float32([[0, 0], [w - _shift(), _shift()], [w - _shift(), h - _shift()], [0, h]])

    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(image_np, M, (w, h),
                                borderMode=cv2.BORDER_CONSTANT,
                                borderValue=(0, 0, 0))

def adjust_brightness(image_np):
    img_tensor = tf.image.random_brightness(image_np, max_delta=BRIGHTNESS_MAX_DELTA)
    img_tensor = tf.clip_by_value(img_tensor, 0, 255)

    return img_tensor.numpy().astype(np.uint8)

def gaussian_blur(image_np):
    k = random.choice([3, 5, 7])
    k = min(k, BLUR_MAX_KERNEL)

    return cv2.GaussianBlur(image_np, (k, k), 0)

def add_noise(image_np):
    img_float = image_np.astype(np.float32) / 255.0
    noise = tf.random.normal(shape=img_float.shape, mean=0.0, stddev=NOISE_FACTOR)
    noisy = tf.clip_by_value(img_float + noise, 0.0, 1.0)

    return (noisy.numpy() * 255).astype(np.uint8)

# ------------------------------------------------
# --- Create the dataset here
# ------------------------------------------------

# Set the operations for the card creation set
operations = [
    perspective_warp_cv,
    adjust_brightness,
    gaussian_blur,
    add_noise,
    ]

def generate_augmented_dataset(augments_per_image=3):
    """
    For each image in CARD_DIR, apply a random subset of operations
    and save the results to AUGMENTED_CARD_DIR.
    """

    AUGMENTED_CARD_DIR.mkdir(parents=True, exist_ok=True)

    image_paths = list(CARD_DIR.glob("*.jpg"))
    print(f"Augmenting {len(image_paths)} images, {augments_per_image}x each...")

    count = 0
    for img_path in image_paths:
        img = cv2.imread(str(img_path))
        if img is None: # image read failed
            continue

        for i in range(augments_per_image):
            augmented = img.copy()

            # Pick a random number of operations
            num_ops = random.randint(1, len(operations))
            chosen_ops = random.sample(operations, num_ops)

            for op in chosen_ops:
                augmented = op(augmented)

            out_name = f"{img_path.stem}_aug{i:02d}.jpg"
            cv2.imwrite(str(CARD_DIR / out_name), augmented)
            count += 1

    print(f"Saved {count} augmented images to {CARD_DIR}")
    return count

# Download

---



In [ ]:
# Clean out old data
for d in [CARD_DIR, NOT_CARD_DIR]:
    if d.exists():
        shutil.rmtree(d)

# Create directories for storage of the data
CARD_DIR.mkdir(parents=True, exist_ok=True)
NOT_CARD_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Run downloads and create the base dataset
cards_downloaded = fetch_scryfall_cards()
non_cards_downloaded = fetch_non_card_images()

Fetching 10 card images from Scryfall...


# Generator dataset

---

In [ ]:
# create the transformed data set
generate_augmented_dataset()

# Camera

In [ ]:
# simple camera function - not going to lie, AI helped me. idk Javascript
# to stop the camera - right like the video -> more controls -> pause key

test_js = Javascript('''
  async function testCamera() {
    try {
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      const video = document.createElement('video');
      video.srcObject = stream;
      video.autoplay = true;
      video.style.width = '320px';
      document.body.appendChild(video);
      return "SUCCESS: camera opened";
    } catch (err) {
      return "ERROR: " + err.name + " - " + err.message;
    }
  }
''')

display(test_js)
result = eval_js('testCamera()')
print(result)

# Display

---

In [ ]:
def show_augmented_samples(num_samples=5):
    """
    Show original cards next to their augmented versions.
    """

    originals = [p for p in CARD_DIR.glob("*.jpg") if "_aug" not in p.stem]
    num_samples = min(num_samples, len(originals))
    samples = random.sample(originals, num_samples)

    fig, axes = plt.subplots(num_samples, 2, figsize=(6, 4 * num_samples))
    if num_samples == 1:
        axes = [axes]

    for row, path in enumerate(samples):
        img = cv2.imread(str(path))
        if img is None:
            continue

        # Left: original
        axes[row][0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row][0].set_title("Original", fontsize=9)

        # Right: find a matching augmented version
        aug_paths = list(CARD_DIR.glob(f"{path.stem}_aug*.jpg"))
        if aug_paths:
            aug_img = cv2.imread(str(random.choice(aug_paths)))
            axes[row][1].imshow(cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB))
            axes[row][1].set_title("Augmented", fontsize=9)
        else:
            axes[row][1].set_title("No augment found", fontsize=9)

        for ax in axes[row]:
            ax.axis('off')

    plt.suptitle("Before and After Augmentation", fontsize=14)
    plt.tight_layout()
    plt.show()

def show_non_card_samples(num_samples=10):
    """
    Display a grid of downloaded non-card images.
    """

    images = list(NOT_CARD_DIR.glob("*.jpg"))
    num_samples = min(num_samples, len(images))
    samples = random.sample(images, num_samples)

    cols = 5
    rows = (num_samples + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
    axes = axes.flatten() if num_samples > 1 else [axes]

    for i, ax in enumerate(axes):
        if i < len(samples):
            img = cv2.imread(str(samples[i]))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                ax.set_title(samples[i].name, fontsize=8)
        ax.axis('off')

    plt.suptitle(f"Non-Card Samples ({len(images)} total)", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
def display_simple():
  """
  Display a random card and its augmented version.
  """

  card_images = list(CARD_DIR.glob("*.jpg"))
  img = Image.open(random.choice(card_images))
  display(img)

  img_np = np.array(img.convert("L"))
  img_np = cv2.resize(img_np, (IMAGE_SIZE, IMAGE_SIZE))
  warped = perspective_warp_cv(img_np)

  display(Image.fromarray(warped))

# Main

---

In [ ]:
# simple display of the before and after of the sample
display_simple()
show_augmented_samples()
show_non_card_samples()


# Data Preperation

---

In [ ]:
# Load images from folders
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="grayscale"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="grayscale"
)

# Rescale pixels from [0, 255] to [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
# Define TinyConv Model

---

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),

    # First Convolution: Find edges/corners
    tf.keras.layers.Conv2D(8, (3, 3), strides=(2, 2), padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Second Convolution: Find card shapes
    tf.keras.layers.Conv2D(16, (3, 3), strides=(2, 2), padding='same', activation='relu'),
    tf.keras.layers.BatchNormalization(),

    # Flatten and Classify (0 = Not Card, 1 = Card)
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# Train and Quantize

---

In [ ]:
# Export as a standard TFLite model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset needed for full integer quantization
def representative_data_gen():
  for input_value, _ in train_ds.take(100):
    yield [input_value]

converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8 = converter.convert()

# Save the model
with open('mtg_detector.tflite', 'wb') as f:
  f.write(tflite_model_int8)

# Convert to C Array

---

In [ ]:
!apt-get update && apt-get install xxd
!xxd -i mtg_detector.tflite > mtg_model_data.cc